# 280K Autoencoder Complete Training Pipeline

This notebook implements the full standard Autoencoder training pipeline on the 280K deduplicated dataset.
It matches the exact directory, checkpointing, and ZIP packaging structure of the VAE 280K pipeline for a fair 1:1 comparison.

In [ ]:
import os
import random
import hashlib
import json
import shutil
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

print("TensorFlow version:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

BASE_DIR = "/kaggle/working/AE_280K"
DIRS = {
    "checkpoints": os.path.join(BASE_DIR, "checkpoints"),
    "logs": os.path.join(BASE_DIR, "logs"),
    "config": os.path.join(BASE_DIR, "config")
}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)


## 1. Load Dataset & Exact Split

In [ ]:
!pip install -q datasets
from datasets import load_dataset
dataset = load_dataset("evilsocket/alucard-sprites")
train_data = dataset["train"]

def image_hash(image):
    pixels = np.asarray(image.convert("RGBA"), dtype=np.uint8)
    return hashlib.sha256(pixels.tobytes()).hexdigest()

UNIQUE_INDICES_PATH = os.path.join(DIRS["config"], "unique_indices.json")

if os.path.exists(UNIQUE_INDICES_PATH):
    with open(UNIQUE_INDICES_PATH, "r") as f:
        all_hashes = json.load(f)
else:
    all_hashes = []
    seen_hashes = set()
    for i, item in enumerate(train_data):
        h = image_hash(item["image"])
        if h not in seen_hashes:
            seen_hashes.add(h)
            all_hashes.append(i)
    with open(UNIQUE_INDICES_PATH, "w") as f:
        json.dump(all_hashes, f)

unique_dataset = train_data.select(all_hashes)

SHUFFLED_INDICES_PATH = os.path.join(DIRS["config"], "shuffled_indices.json")
if os.path.exists(SHUFFLED_INDICES_PATH):
    with open(SHUFFLED_INDICES_PATH, "r") as f:
        shuffled_indices = json.load(f)
else:
    shuffled_indices = list(range(len(unique_dataset)))
    random.seed(42)
    random.shuffle(shuffled_indices)
    with open(SHUFFLED_INDICES_PATH, "w") as f:
        json.dump(shuffled_indices, f)

unique_dataset = unique_dataset.select(shuffled_indices)

TOTAL_CLEAN = len(unique_dataset)
TRAIN_SIZE = int(round(TOTAL_CLEAN * 0.90))
VAL_SIZE = int(round(TOTAL_CLEAN * 0.05))
TEST_SIZE = TOTAL_CLEAN - TRAIN_SIZE - VAL_SIZE

train_dataset = unique_dataset.select(range(0, TRAIN_SIZE))
val_dataset = unique_dataset.select(range(TRAIN_SIZE, TRAIN_SIZE + VAL_SIZE))
test_dataset = unique_dataset.select(range(TRAIN_SIZE + VAL_SIZE, len(unique_dataset)))

print("Training:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))


## 2. Lazy tf.data Pipeline

In [ ]:
def data_generator(hf_dataset):
    for item in hf_dataset:
        image = item["image"].convert("RGBA")
        image_np = np.array(image, dtype=np.float32) / 255.0
        yield (image_np, image_np)

GLOBAL_BATCH_SIZE = 64

def create_tf_dataset(hf_dataset):
    return tf.data.Dataset.from_generator(
        lambda: data_generator(hf_dataset),
        output_signature=(
            tf.TensorSpec(shape=(128, 128, 4), dtype=tf.float32),
            tf.TensorSpec(shape=(128, 128, 4), dtype=tf.float32)
        )
    )

train_ds = create_tf_dataset(train_dataset).batch(GLOBAL_BATCH_SIZE).repeat().prefetch(tf.data.AUTOTUNE)
val_ds = create_tf_dataset(val_dataset).batch(GLOBAL_BATCH_SIZE).repeat().prefetch(tf.data.AUTOTUNE)
test_ds = create_tf_dataset(test_dataset).batch(GLOBAL_BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

test_originals = []
for images, _ in test_ds.take(2):
    test_originals.append(images.numpy())
test_originals = np.concatenate(test_originals, axis=0)
print("Pipelines ready.")


## 3. Autoencoder Architecture

In [ ]:
strategy = tf.distribute.MirroredStrategy()
print("Number of devices:", strategy.num_replicas_in_sync)

with strategy.scope():
    latent_dim = 256

    # Encoder
    encoder_inputs = keras.Input(shape=(128, 128, 4))
    x = layers.Conv2D(32, 3, activation="relu", padding="same", strides=2)(encoder_inputs)
    x = layers.Conv2D(64, 3, activation="relu", padding="same", strides=2)(x)
    x = layers.Conv2D(128, 3, activation="relu", padding="same", strides=2)(x)
    x = layers.Conv2D(256, 3, activation="relu", padding="same", strides=2)(x)
    x = layers.Flatten()(x)
    latent_outputs = layers.Dense(latent_dim, name="latent_outputs")(x)
    encoder = keras.Model(encoder_inputs, latent_outputs, name="encoder")

    # Decoder
    latent_inputs = keras.Input(shape=(latent_dim,))
    x = layers.Dense(8 * 8 * 256, activation="relu")(latent_inputs)
    x = layers.Reshape((8, 8, 256))(x)
    x = layers.Conv2DTranspose(128, 4, activation="relu", padding="same", strides=2)(x)
    x = layers.Conv2DTranspose(64, 4, activation="relu", padding="same", strides=2)(x)
    x = layers.Conv2DTranspose(32, 4, activation="relu", padding="same", strides=2)(x)
    decoder_outputs = layers.Conv2DTranspose(4, 4, activation="sigmoid", padding="same", strides=2)(x)
    decoder = keras.Model(latent_inputs, decoder_outputs, name="decoder")

    # Full Autoencoder
    ae_inputs = keras.Input(shape=(128, 128, 4))
    encoded = encoder(ae_inputs)
    decoded = decoder(encoded)
    autoencoder = keras.Model(ae_inputs, decoded, name="autoencoder")
    
    autoencoder.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss="mse")

encoder.summary()
decoder.summary()
autoencoder.summary()


## 4. Resumable Training Pipeline

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=7, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint(filepath=os.path.join(DIRS["checkpoints"], "latest.weights.h5"), save_weights_only=True),
    keras.callbacks.ModelCheckpoint(filepath=os.path.join(DIRS["checkpoints"], "best.weights.h5"), save_best_only=True, monitor="val_loss", save_weights_only=True),
    keras.callbacks.CSVLogger(os.path.join(DIRS["logs"], "training_log.csv"), append=True)
]

EPOCHS = 50
STEPS_PER_EPOCH = TRAIN_SIZE // GLOBAL_BATCH_SIZE
VAL_STEPS = VAL_SIZE // GLOBAL_BATCH_SIZE

# Resume checkpoint logic
initial_epoch = 0
latest_checkpoint = os.path.join(DIRS["checkpoints"], "latest.weights.h5")
log_path = os.path.join(DIRS["logs"], "training_log.csv")

if os.path.exists(latest_checkpoint):
    try:
        print(f"Found existing checkpoint at {latest_checkpoint}. Loading weights to resume...")
        autoencoder.load_weights(latest_checkpoint)
        
        if os.path.exists(log_path):
            df = pd.read_csv(log_path)
            if len(df) > 0 and 'epoch' in df.columns:
                initial_epoch = int(df['epoch'].iloc[-1]) + 1
                print(f"Resuming training from epoch {initial_epoch}...")
    except Exception as e:
        print(f"Could not load checkpoint, starting from scratch. Error: {e}")

history = autoencoder.fit(
    train_ds,
    steps_per_epoch=STEPS_PER_EPOCH,
    epochs=EPOCHS,
    validation_data=val_ds,
    validation_steps=VAL_STEPS,
    callbacks=callbacks,
    initial_epoch=initial_epoch
)

print("Saving final weights and models to working directory...")
encoder.save_weights(os.path.join(BASE_DIR, "encoder_280k_final.weights.h5"))
decoder.save_weights(os.path.join(BASE_DIR, "decoder_280k_final.weights.h5"))
encoder.save(os.path.join(BASE_DIR, "encoder_280k_final.keras"))
decoder.save(os.path.join(BASE_DIR, "decoder_280k_final.keras"))
print("Final weights and models saved successfully!")


## 5. Evaluation

In [ ]:
hist = pd.read_csv(os.path.join(DIRS["logs"], "training_log.csv"))

plt.figure(figsize=(8, 5))
plt.plot(hist['loss'], label='Train MSE Loss')
plt.plot(hist['val_loss'], label='Val MSE Loss')
plt.title('Autoencoder Training Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.legend()
plt.savefig(os.path.join(BASE_DIR, "loss_plot.png"), bbox_inches='tight')
plt.show()


In [ ]:
random.seed(42)
random_indices = random.sample(range(len(test_originals)), 5)
test_reconstructions = autoencoder.predict(test_originals)

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i, idx in enumerate(random_indices):
    axes[0, i].imshow(test_originals[idx])
    axes[0, i].set_title("Original")
    axes[0, i].axis("off")
    axes[1, i].imshow(test_reconstructions[idx])
    axes[1, i].set_title("AE Reconstructed")
    axes[1, i].axis("off")

plt.suptitle("Test Set Reconstructions", fontsize=16)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, "reconstruction_samples.png"), bbox_inches='tight')
plt.show()


## 6. Package Outputs for Download

In [ ]:
import shutil
from IPython.display import FileLink

zip_path = "/kaggle/working/AE_280K_Outputs"
shutil.make_archive(zip_path, 'zip', BASE_DIR)

print(f"Created {zip_path}.zip!")
print("Click the link below to download your files:")
FileLink(r'AE_280K_Outputs.zip')
